In [1]:
# Paired t-test: REMNET ensemble vs individual CNNs (CLASSIFICATION ONLY)
# Structure adapted from the supplied paired_t_test_accuracy.ipynb.
# Verified full-precision seed-wise values are embedded below, extracted from
# Stage22/Acc_Probability/*/*_Performance_Per_Seed.csv. Their displayed summaries
# match the outputs of the latest uploaded classification checkpoint notebooks.
# No dataset loading, retraining, sustainability or CodeCarbon measurements.

import scipy.stats as stats
import math
import numpy as np
import csv
from pathlib import Path

# Sample size (seeds) and significance threshold
SEEDS = [7, 49, 112, 345, 476]
n = len(SEEDS)
alpha = 0.05

# All values in PERCENT; computations use full per-seed precision (not rounded means).
# Comparisons are matched by the same seed ID; difference = REMNET - individual CNN.
metrics_data = {'REMNET': {'accuracy': [94.60043197, 94.81641468999999, 93.73650108, 94.16846652000001, 93.30453564],
            'macro_precision': [94.61815637, 94.83383531, 93.57464925, 94.57264949, 93.50701672],
            'macro_recall': [93.37591252, 93.44883659, 91.96980581000001, 92.43081895, 91.42886655],
            'macro_f1': [93.79139613000001, 93.96367105, 92.49409419999999, 93.1294169, 92.00232038]},
 'ResNet-50': {'accuracy': [90.49676025917927,
                            92.0086393088553,
                            93.9524838012959,
                            90.06479481641469,
                            91.5766738660907],
               'macro_precision': [90.3510351129169,
                                   92.27214939013307,
                                   93.54973634531673,
                                   89.83981234033695,
                                   91.67791800596619],
               'macro_recall': [89.97892078518154,
                                90.63397780889014,
                                93.38860416791542,
                                88.75918882658259,
                                89.20809328796554],
               'macro_f1': [89.81099986381676,
                            91.17630041682386,
                            93.45788011435054,
                            89.00002121230519,
                            89.55418285046443]},
 'MobileNetV2': {'accuracy': [90.92872570194385,
                              90.92872570194385,
                              90.06479481641469,
                              89.8488120950324,
                              91.14470842332614],
                 'macro_precision': [91.90020351302773,
                                     90.7814808559689,
                                     91.75678359248339,
                                     90.90721471058283,
                                     92.20538109943858],
                 'macro_recall': [88.64506979588573,
                                  89.75422453342662,
                                  87.58659753237747,
                                  87.71505837544512,
                                  89.02624216533104],
                 'macro_f1': [89.36172034524205,
                              90.16443150140691,
                              88.48711251457246,
                              88.60013708243336,
                              89.88096764008796]},
 'EfficientNet-Lite0': {'accuracy': [90.28077753779698,
                                     87.47300215982722,
                                     90.71274298056156,
                                     90.49676025917927,
                                     90.49676025917927],
                        'macro_precision': [90.51382904852865,
                                            87.5412244857963,
                                            90.87655044928574,
                                            90.9418445926773,
                                            91.07915252462692],
                        'macro_recall': [88.72598985647127,
                                         85.230361363211,
                                         88.47145868477463,
                                         88.03541661824474,
                                         88.66624292377145],
                        'macro_f1': [89.38259968505592,
                                     85.66293768172865,
                                     89.00318036783541,
                                     88.61533234158378,
                                     89.33735588268542]}}

# Keep the same model/metric order as the reference statistical notebook.
model_order = ["ResNet-50", "MobileNetV2", "EfficientNet-Lite0"]
metric_order = ["Accuracy", "Macro-Precision", "Macro-Recall", "Macro-F1"]
metric_keys = {
    "Accuracy": "accuracy",
    "Macro-Precision": "macro_precision",
    "Macro-Recall": "macro_recall",
    "Macro-F1": "macro_f1",
}

# Outputs are written in the Kaggle working directory when available,
# otherwise alongside the notebook's current working directory.
OUT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUT.mkdir(parents=True, exist_ok=True)
result_file = OUT / "REMNET_Classification_Paired_T_Test_Results.csv"

# Holm step-down FWER correction, no additional third-party dependency.
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    m = len(p_values)
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p_values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted

# Optional scientific-notation formatting, mirroring the reference notebook.
def format_to_unicode_power(value):
    if value == 0:
        return "0"
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    superscripts = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")
    return f"{mantissa:.2f} × 10{str(exponent).translate(superscripts)}"

print("Classification statistical comparison: REMNET vs. individual CNNs")
print(f"Same seed order: {SEEDS}; two-sided paired t-test; n={n}; df={n-1}; alpha={alpha}")
print("Pairwise difference = REMNET - CNN, measured in percentage points")
print("Cohen's dz = mean(paired differences) / sample SD(paired differences)")
print("Holm correction: across the three CNN comparisons separately for each metric.")

records = []
for metric in metric_order:
    metric_key = metric_keys[metric]
    remnet = np.asarray(metrics_data["REMNET"][metric_key], dtype=float)
    assert len(remnet) == n and np.all(np.isfinite(remnet))
    for model_name in model_order:
        cnn = np.asarray(metrics_data[model_name][metric_key], dtype=float)
        assert len(cnn) == n and np.all(np.isfinite(cnn))
        differences = remnet - cnn
        mean_diff = float(np.mean(differences))
        sd_diff = float(np.std(differences, ddof=1))
        sem_diff = sd_diff / math.sqrt(n)
        if sd_diff == 0:
            t_stat = float('inf') if mean_diff > 0 else float('-inf') if mean_diff < 0 else float('nan')
            p_value = 0.0 if mean_diff != 0 else 1.0
            ci_low = ci_high = mean_diff
            cohen_dz = float('inf') if mean_diff > 0 else float('-inf') if mean_diff < 0 else 0.0
        else:
            t_result = stats.ttest_rel(remnet, cnn, alternative="two-sided")
            t_stat, p_value = float(t_result.statistic), float(t_result.pvalue)
            ci_low, ci_high = stats.t.interval(
                confidence=0.95, df=n - 1, loc=mean_diff, scale=sem_diff
            )
            cohen_dz = mean_diff / sd_diff
        records.append({
            "metric": metric,
            "comparison": f"REMNET vs {model_name}",
            "model": model_name,
            "n_paired_seeds": n,
            "seeds": ";".join(str(s) for s in SEEDS),
            "REMNET_mean_pct": float(np.mean(remnet)),
            "REMNET_sample_SD_pct": float(np.std(remnet, ddof=1)),
            "CNN_mean_pct": float(np.mean(cnn)),
            "CNN_sample_SD_pct": float(np.std(cnn, ddof=1)),
            "paired_mean_difference_percentage_points": mean_diff,
            "paired_difference_sample_SD_percentage_points": sd_diff,
            "CI_95_lower_percentage_points": float(ci_low),
            "CI_95_upper_percentage_points": float(ci_high),
            "paired_t_statistic": t_stat,
            "degrees_of_freedom": n - 1,
            "paired_cohens_dz": cohen_dz,
            "p_value_two_sided": p_value,
        })

# Primary: correct the 3 pairwise REMNET comparisons within each metric.
for metric in metric_order:
    group = [r for r in records if r["metric"] == metric]
    group_pvals = holm_adjust([r["p_value_two_sided"] for r in group])
    for r, adjusted in zip(group, group_pvals):
        r["holm_adjusted_p_within_metric_3"] = float(adjusted)
        r["significant_holm_within_metric_0_05"] = "Yes" if adjusted < alpha else "No"

# Sensitivity check: conservative Holm correction across all 12 comparisons.
all_12 = holm_adjust([r["p_value_two_sided"] for r in records])
for r, adjusted in zip(records, all_12):
    r["holm_adjusted_p_all_12"] = float(adjusted)
    r["significant_holm_all_12_0_05"] = "Yes" if adjusted < alpha else "No"

print("\nDescriptive statistics: five-seed mean ± sample SD (%)")
for metric in metric_order:
    print(f"  {metric}:")
    for model_name in ["REMNET", *model_order]:
        arr = np.asarray(metrics_data[model_name][metric_keys[metric]], dtype=float)
        print(f"    {model_name:20s}: {arr.mean():.4f} ± {arr.std(ddof=1):.4f}")

print("\nPaired comparisons (Holm correction within each metric):")
print(f"{'Model':<20} | {'Metric':<16} | {'Mean diff (pp)':>14} | {'95% CI (pp)':<20} | {'dz':>7} | {'p':>12} | {'p Holm':>12} | {'Decision'}")
print("-" * 137)
for r in records:
    ci = f"({r['CI_95_lower_percentage_points']:.3f}, {r['CI_95_upper_percentage_points']:.3f})"
    print(f"{r['model']:<20} | {r['metric']:<16} | {r['paired_mean_difference_percentage_points']:>14.4f} | {ci:<20} | {r['paired_cohens_dz']:>7.3f} | {r['p_value_two_sided']:>12.5g} | {r['holm_adjusted_p_within_metric_3']:>12.5g} | {r['significant_holm_within_metric_0_05']}")

# Save all requested quantities in ONE standalone CSV.
headers = list(records[0].keys())
with open(result_file, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=headers)
    writer.writeheader()
    writer.writerows(records)
print(f"\nCSV saved: {result_file}")
print("Interpretation: tests are exploratory with n=5 paired seeds; do not infer")
print("superiority solely from raw p-values, particularly across multiple metrics.")


Classification statistical comparison: REMNET vs. individual CNNs
Same seed order: [7, 49, 112, 345, 476]; two-sided paired t-test; n=5; df=4; alpha=0.05
Pairwise difference = REMNET - CNN, measured in percentage points
Cohen's dz = mean(paired differences) / sample SD(paired differences)
Holm correction: across the three CNN comparisons separately for each metric.

Descriptive statistics: five-seed mean ± sample SD (%)
  Accuracy:
    REMNET              : 94.1253 ± 0.6185
    ResNet-50           : 91.6199 ± 1.5226
    MobileNetV2         : 90.5832 ± 0.5836
    EfficientNet-Lite0  : 89.8920 ± 1.3609
  Macro-Precision:
    REMNET              : 94.2213 ± 0.6294
    ResNet-50           : 91.5381 ± 1.4915
    MobileNetV2         : 91.5102 ± 0.6306
    EfficientNet-Lite0  : 90.1905 ± 1.4957
  Macro-Recall:
    REMNET              : 92.5308 ± 0.8798
    ResNet-50           : 90.3938 ± 1.8218
    MobileNetV2         : 88.5454 ± 0.9098
    EfficientNet-Lite0  : 87.8259 ± 1.4760
  Macro-F1:
 